In [9]:
import numpy as np
import xcdat as xc
import scipy
import sys
import matplotlib as mpl
import matplotlib.pyplot as plt 
from cdo import *   # python version
import scipy.stats as stats
import os

sys.path.append('../../functions/')
from lag_linregress import *
from monthly_departures import *
from MCA import *
cdo = Cdo()

import shutil
import tempfile
from pathlib import Path

In [ ]:
# from pathlib import Path
# import subprocess

# # Set these two paths for the remote machine.
# maria_root = Path("/rugenstein-archive/mariarug/model_output/MMLEA")
# lowres_template = Path('../../amip/data/lowres_template.nc')

# model = "CNRM-CM6-1"
# experiment = "amip-hist"
# members = [f"r{i}i1p1f2" for i in range(1, 11)]

# output_dir = Path("./data/CNRM-CM6-1_amip-hist_N_128x64_200003-201412")
# output_dir.mkdir(parents=True, exist_ok=True)

# for member in members:
#     source_dir = maria_root / model / experiment

#     rsdt = source_dir / f"rsdt_Amon_{model}_{experiment}_{member}_gr_187001-201412.nc"
#     rlut = source_dir / f"rlut_Amon_{model}_{experiment}_{member}_gr_187001-201412.nc"
#     rsut = source_dir / f"rsut_Amon_{model}_{experiment}_{member}_gr_187001-201412.nc"

#     output = output_dir / f"N_Amon_{model}_{experiment}_{member}_128x64_187001-201412.nc"

#     command = [
#         "cdo", "-L", "-O",
#         "-f", "nc4c", "-z", "zip_4",
#         f"remapcon,{lowres_template}",
#         # "-seldate,2000-03-01,2014-12-31",
#         "-expr,N=rsdt-rlut-rsut",
#         "-merge",
#         str(rsdt), str(rlut), str(rsut),
#         str(output),
#     ]

#     print(" ".join(command))
#     subprocess.run(command, check=True)

# subprocess.run(["du", "-h", "-d", "1", str(output_dir)], check=True)

In [ ]:
# from pathlib import Path
# import subprocess
# import xarray as xr

# # Existing per-member outputs from the CDO cell you just ran.
# output_dir = Path("./data/CNRM-CM6-1_amip-hist_N_128x64_200003-201412")
# model = "CNRM-CM6-1"
# experiment = "amip-hist"
# period = "187001-201412"

# members = [f"r{i}i1p1f2" for i in range(1, 11)]
# member_files = [
#     output_dir / f"N_Amon_{model}_{experiment}_{member}_128x64_{period}.nc"
#     for member in members
# ]

# missing = [path for path in member_files if not path.exists()]
# if missing:
#     raise FileNotFoundError(f"Missing member files:\n" + "\n".join(map(str, missing)))

# combined_file = output_dir / f"N_Amon_{model}_{experiment}_members_128x64_{period}.nc"

# # The member order follows Senne's convention:
# # member index 0 = r1i1p1f2, member index 1 = r2i1p1f2, ..., member index 9 = r10i1p1f2.
# datasets = []
# for member, path in zip(members, member_files):
#     with xr.open_dataset(path) as ds:
#         datasets.append(ds[["N"]].load().expand_dims(member=[member]))

# combined = xr.concat(datasets, dim="member")
# combined["member"].attrs = {
#     "long_name": "CMIP6 ensemble member identifier",
#     "comment": "Member order follows the numerical CMIP realization number.",
# }
# combined.attrs.update({
#     "model": model,
#     "experiment": experiment,
#     "N_definition": "rsdt - rlut - rsut",
#     "regridding": "CDO remapcon to lowres_template.nc",
# })

# combined.to_netcdf(
#     combined_file,
#     format="NETCDF4",
#     encoding={
#         "N": {
#             "zlib": True,
#             "complevel": 4,
#             "dtype": "float32",
#             "chunksizes": (1, 12, 64, 128),
#         }
#     },
# )

# combined.close()

# subprocess.run(["ncdump", "-h", str(combined_file)], check=True)
# subprocess.run(["du", "-h", str(combined_file)], check=True)

In [ ]:
test = xc.open_dataset('../data/CNRM-CM6-1_amip-hist_N_128x64_200003-201412/N_Amon_CNRM-CM6-1_amip-hist_members_128x64_187001-201412.nc')

In [ ]:
test['N'][0][0].plot()

In [2]:
# Cell 1 — Imports and configuration

from pathlib import Path
from collections import defaultdict
import csv
import re
import subprocess
import tempfile

import numpy as np
from netCDF4 import Dataset


# Maria's archive root.
maria_root = Path("/rugenstein-archive/mariarug/model_output/MMLEA")

# Target 128x64 grid.
lowres_template = Path("/scratch/leiff/amip/data/lowres_template.nc")

# Use the detailed CSV rather than maria_processing_tasks.json:
# it includes both ordinary one-file records and time-split ("ambiguous") records.
catalog_path = Path("./maria_catalog/maria_catalog.csv")

# Final output layout:
# ./data/AMIP/<model>/N_Amon_<model>_<experiment>_combined_187001-201412.nc
output_root = Path("./data/AMIP")

# All output files are restricted to this common period.
start_date = "1870-01-01"
end_date = "2014-12-31"
output_period = "187001-201412"

# Leave False for now: finished combined files will be skipped rather than replaced.
overwrite = False

In [3]:
# Cell 2 — Senne overlap and Maria task selection
# Models for which Senne has processed monthly tas.
# This determines the requested overlap; Maria supplies rsdt, rlut, and rsut.

senne_tas_models = {
    "amip-hist": {
        "BCC-CSM2-MR",
        "CAMS-CSM1-0",
        "CESM2",
        "CIESM",
        "CNRM-CM6-1",
        "CNRM-CM6-1-HR",
        "CNRM-ESM2-1",
        "CanESM5",
        "FGOALS-f3-L",
        "FGOALS-g3",
        "FIO-ESM-2-0",
        "IITM-ESM",
        "IPSL-CM6A-LR",
        "MIROC6",
        "MRI-ESM2-0",
        "TaiESM1",
    },
    "amip-piForcing": {
        "CESM2",
        "CNRM-CM6-1",
        "CanESM5",
        "HadGEM3-GC31-LL",
        "IPSL-CM6A-LR",
        "MIROC6",
        "MRI-ESM2-0",
        "TaiESM1",
    },
}

radiation_variables = ("rsdt", "rlut", "rsut")

with catalog_path.open(newline="") as handle:
    maria_catalog = list(csv.DictReader(handle))

# Keep a member if Maria has all three radiation components.
# We intentionally do NOT require Maria tas here: Senne's processed tas is the
# matching tas product, while Maria is only needed to make spatial N.
tasks_by_model_experiment = defaultdict(list)

for row in maria_catalog:
    model = row["model"]
    experiment = row["experiment"]

    if experiment not in senne_tas_models:
        continue
    if model not in senne_tas_models[experiment]:
        continue
    if any(not row[f"{variable}_path"] for variable in radiation_variables):
        continue

    tasks_by_model_experiment[(model, experiment)].append(row)

print("Maria radiation-data overlap with Senne's processed monthly tas:")
for (model, experiment), tasks in sorted(tasks_by_model_experiment.items()):
    print(f"  {experiment:15s}  {model:20s}  {len(tasks)} member(s)")

Maria radiation-data overlap with Senne's processed monthly tas:
  amip-hist        BCC-CSM2-MR           1 member(s)
  amip-hist        CAMS-CSM1-0           3 member(s)
  amip-hist        CESM2                 3 member(s)
  amip-piForcing   CESM2                 1 member(s)
  amip-hist        CIESM                 3 member(s)
  amip-hist        CNRM-CM6-1            10 member(s)
  amip-piForcing   CNRM-CM6-1            1 member(s)
  amip-hist        CNRM-CM6-1-HR         1 member(s)
  amip-hist        CNRM-ESM2-1           1 member(s)
  amip-hist        CanESM5               10 member(s)
  amip-piForcing   CanESM5               3 member(s)
  amip-hist        FGOALS-f3-L           3 member(s)
  amip-hist        FGOALS-g3             4 member(s)
  amip-hist        FIO-ESM-2-0           3 member(s)
  amip-piForcing   HadGEM3-GC31-LL       1 member(s)
  amip-hist        IITM-ESM              1 member(s)
  amip-hist        IPSL-CM6A-LR          3 member(s)
  amip-piForcing   IPSL-CM6A-LR 

In [11]:
# Cell 3 — Helpers for time chunks and CDO processing

def realization_number(member_name):
    """Numerical ordering: r1, r2, ..., r10 rather than alphabetical order."""
    return int(re.match(r"r(\d+)", member_name).group(1))

def cdo_input_stream(paths):
    """
    Wrap each variable stream in CDO argument-group brackets.

    This lets CDO distinguish:
      merge(mergetime(rsdt chunks),
            mergetime(rlut chunks),
            mergetime(rsut chunks))
    """
    if len(paths) == 1:
        return ["[", str(paths[0]), "]"]

    return [
        "[",
        "-mergetime",
        *map(str, paths),
        "]",
    ]

def month_number(yyyymm):
    """Convert YYYYMM to a monotonically increasing month number."""
    year = int(yyyymm[:4])
    month = int(yyyymm[4:6])
    return year * 12 + month - 1


def member_paths(row, variable):
    """
    Return a variable's source files in chronological order.

    A normal Maria record has one path.
    A catalog 'ambiguous' record contains semicolon-separated, sequential
    time chunks. These are safe to use when they form one continuous series.
    """
    relative_paths = [
        path for path in row[f"{variable}_path"].split(";")
        if path
    ]

    paths_with_periods = []
    for relative_path in relative_paths:
        match = re.search(r"_(\d{6})-(\d{6})\.nc$", relative_path)
        if not match:
            raise ValueError(f"Could not read date range from {relative_path}")

        start, end = match.groups()
        paths_with_periods.append(
            (month_number(start), month_number(end), maria_root / relative_path.lstrip("./"))
        )

    paths_with_periods.sort()

    # Confirm that adjacent source files are consecutive monthly chunks,
    # rather than duplicate versions or competing grids.
    for (_, previous_end, _), (next_start, _, _) in zip(
        paths_with_periods[:-1],
        paths_with_periods[1:],
    ):
        if next_start != previous_end + 1:
            raise ValueError(
                f"{row['model']} {row['experiment']} {row['member']} {variable}: "
                "source chunks are not continuous in time."
            )

    # Every source collection must cover the desired common output period.
    if paths_with_periods[0][0] > month_number("187001"):
        raise ValueError(f"{variable} starts after 1870-01 for {row['member']}")
    if paths_with_periods[-1][1] < month_number("201412"):
        raise ValueError(f"{variable} ends before 2014-12 for {row['member']}")

    paths = [path for _, _, path in paths_with_periods]

    missing = [str(path) for path in paths if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing Maria source file(s):\n" + "\n".join(missing))

    return paths


# def create_member_n(row, temporary_n_file):
#     """
#     Merge any sequential time chunks, calculate N, clip to 1870–2014,
#     and conservatively regrid to the low-resolution template.
#     """
#     rsdt_paths = member_paths(row, "rsdt")
#     rlut_paths = member_paths(row, "rlut")
#     rsut_paths = member_paths(row, "rsut")

#     def grouped_stream(paths):
#         # CDO needs [ ... ] to distinguish the three variable streams when
#         # one or more of them uses the multiple-input mergetime operator.
#         if len(paths) == 1:
#             return ["[", str(paths[0]), "]"]

#         return [
#             "[",
#             "-mergetime",
#             *map(str, paths),
#             "]",
#         ]

#     command = [
#         "cdo",
#         "-L",
#         "-O",
#         "-f", "nc4c",
#         "-z", "zip_4",
#         f"remapcon,{lowres_template}",
#         f"-seldate,{start_date},{end_date}",
#         "-expr,N=rsdt-rlut-rsut",
#         "-merge",
#         *grouped_stream(rsdt_paths),
#         *grouped_stream(rlut_paths),
#         *grouped_stream(rsut_paths),
#         str(temporary_n_file),
#     ]

#     print(f"    CDO: {row['member']}")

#     result = subprocess.run(command, text=True, capture_output=True)

#     if result.returncode != 0:
#         print("\nCDO command:")
#         print(" ".join(command))
#         print("\nCDO stderr:")
#         print(result.stderr)
#         raise RuntimeError(
#             f"CDO failed for {row['model']} / "
#             f"{row['experiment']} / {row['member']}"
#         )


def create_member_n(row, temporary_n_file):
    """
    Merge sequential time chunks for rsdt, rlut, and rsut,
    calculate N = rsdt - rlut - rsut,
    clip to 1870–2014,
    and conservatively regrid to the low-resolution template.
    """
    rsdt_paths = member_paths(row, "rsdt")
    rlut_paths = member_paths(row, "rlut")
    rsut_paths = member_paths(row, "rsut")

    # Temporary files for the three merged variables
    tmp_dir = Path(tempfile.mkdtemp(prefix="cdo_inputs_"))

    try:
        rsdt_merged = tmp_dir / "rsdt.nc"
        rlut_merged = tmp_dir / "rlut.nc"
        rsut_merged = tmp_dir / "rsut.nc"
        merged = tmp_dir / "merged.nc"

        def mergetime(paths, output):
            command = [
                "cdo",
                "-L",
                "-O",
                "mergetime",
                *map(str, paths),
                str(output),
            ]

            result = subprocess.run(
                command,
                text=True,
                capture_output=True,
            )

            if result.returncode != 0:
                print("\nCDO command:")
                print(" ".join(command))
                print("\nCDO stderr:")
                print(result.stderr)
                raise RuntimeError(
                    f"CDO mergetime failed for {row['model']} / "
                    f"{row['experiment']} / {row['member']}"
                )

        # Merge each variable's time chunks
        mergetime(rsdt_paths, rsdt_merged)
        mergetime(rlut_paths, rlut_merged)
        mergetime(rsut_paths, rsut_merged)

        # Put the three variables into one file
        command = [
            "cdo",
            "-L",
            "-O",
            "merge",
            str(rsdt_merged),
            str(rlut_merged),
            str(rsut_merged),
            str(merged),
        ]

        result = subprocess.run(
            command,
            text=True,
            capture_output=True,
        )

        if result.returncode != 0:
            print("\nCDO command:")
            print(" ".join(command))
            print("\nCDO stderr:")
            print(result.stderr)
            raise RuntimeError(
                f"CDO merge failed for {row['model']} / "
                f"{row['experiment']} / {row['member']}"
            )

        # Calculate N, select dates, and remap
        command = [
            "cdo",
            "-L",
            "-O",
            "-f", "nc4c",
            "-z", "zip_4",
            f"remapcon,{lowres_template}",
            f"-seldate,{start_date},{end_date}",
            "-expr,N=rsdt-rlut-rsut",
            str(merged),
            str(temporary_n_file),
        ]

        print(f"    CDO: {row['member']}")

        result = subprocess.run(
            command,
            text=True,
            capture_output=True,
        )

        if result.returncode != 0:
            print("\nCDO command:")
            print(" ".join(command))
            print("\nCDO stderr:")
            print(result.stderr)
            raise RuntimeError(
                f"CDO failed for {row['model']} / "
                f"{row['experiment']} / {row['member']}"
            )

    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

In [5]:
# Cell 4 — Write members one at a time into a combined NetCDF

def copy_member_to_combined(source_file, combined_file, member_index, member_name):
    """
    Copy a single temporary N field into its proper location in the final file.

    The final file contains only:
      - member(member): CMIP6 strings, such as r1i1p1f2
      - time, lat, lon
      - N(member, time, lat, lon)

    This avoids keeping a model's entire ensemble in memory.
    """
    with Dataset(source_file, "r") as source:
        source_n = source.variables["N"]
        source_dimensions = source_n.dimensions

        # The first member creates the final file and its shared coordinates.
        if not combined_file.exists():
            with Dataset(combined_file, "w", format="NETCDF4") as destination:
                destination.createDimension("member", None)

                for dimension in source_dimensions:
                    destination.createDimension(
                        dimension,
                        len(source.dimensions[dimension]),
                    )

                # Copy coordinate variables, normally time, lat, and lon.
                for dimension in source_dimensions:
                    source_coordinate = source.variables[dimension]

                    destination_coordinate = destination.createVariable(
                        dimension,
                        source_coordinate.dtype,
                        (dimension,),
                    )
                    destination_coordinate[:] = source_coordinate[:]

                    for attribute in source_coordinate.ncattrs():
                        destination_coordinate.setncattr(
                            attribute,
                            source_coordinate.getncattr(attribute),
                        )

                # The named coordinate that Senne's files lack.
                member_coordinate = destination.createVariable(
                    "member",
                    str,
                    ("member",),
                )
                member_coordinate.long_name = "CMIP6 ensemble member identifier"
                member_coordinate.comment = (
                    "Member order follows the numerical CMIP realization number."
                )

                # Write N compressed, with chunks that keep one member and one
                # year of monthly data together.
                n_output = destination.createVariable(
                    "N",
                    "f4",
                    ("member",) + source_dimensions,
                    zlib=True,
                    complevel=4,
                    chunksizes=(
                        1,
                        min(12, len(source.dimensions["time"])),
                        len(source.dimensions["lat"]),
                        len(source.dimensions["lon"]),
                    ),
                    fill_value=np.float32(np.nan),
                )

                for attribute in source_n.ncattrs():
                    if attribute != "_FillValue":
                        n_output.setncattr(
                            attribute,
                            source_n.getncattr(attribute),
                        )

                n_output.long_name = "Net top-of-atmosphere radiation"
                n_output.N_definition = "rsdt - rlut - rsut"
                n_output.regridding = "CDO remapcon to lowres_template.nc"

        # Add this member, then close the temporary source file before moving on.
        with Dataset(combined_file, "a") as destination:
            destination.variables["member"][member_index] = member_name
            destination.variables["N"][member_index, :, :, :] = source_n[:, :, :]

In [12]:
# Cell 5 — Run every selected model and experiment
for (model, experiment), tasks in sorted(tasks_by_model_experiment.items()):
    # Use Senne's numerical ensemble ordering.
    tasks = sorted(tasks, key=lambda row: realization_number(row["member"]))

    model_output_dir = output_root / model
    model_output_dir.mkdir(parents=True, exist_ok=True)

    final_file = (
        model_output_dir
        / f"N_Amon_{model}_{experiment}_combined_{output_period}.nc"
    )
    partial_file = final_file.with_suffix(".partial.nc")

    # Do not accidentally replace a finished result.
    if final_file.exists() and not overwrite:
        print(f"Skipping existing file: {final_file}")
        continue

    # A partial file is only used while this model/experiment is processing.
    # It is renamed to the final filename only after every member succeeds.
    if partial_file.exists():
        partial_file.unlink()

    print(f"\n{model} / {experiment}: {len(tasks)} member(s)")

    try:
        with tempfile.TemporaryDirectory(prefix="cdo_N_") as temporary_directory:
            temporary_directory = Path(temporary_directory)

            for member_index, row in enumerate(tasks):
                temporary_n_file = temporary_directory / f"{row['member']}_N.nc"

                # Create N for one member, copy it to the combined file, and
                # delete the temporary file before processing the next member.
                create_member_n(row, temporary_n_file)

                copy_member_to_combined(
                    temporary_n_file,
                    partial_file,
                    member_index,
                    row["member"],
                )

                temporary_n_file.unlink()

        # This rename marks the product as complete.
        partial_file.replace(final_file)

        print(f"  Saved: {final_file}")
        subprocess.run(["du", "-h", str(final_file)], check=True)

    except Exception:
        print(f"  FAILED: incomplete output retained at {partial_file}")
        raise


Skipping existing file: data/AMIP/BCC-CSM2-MR/N_Amon_BCC-CSM2-MR_amip-hist_combined_187001-201412.nc
Skipping existing file: data/AMIP/CAMS-CSM1-0/N_Amon_CAMS-CSM1-0_amip-hist_combined_187001-201412.nc

CESM2 / amip-hist: 3 member(s)
    CDO: r1i1p1f1
    CDO: r2i1p1f1
    CDO: r3i1p1f1
  Saved: data/AMIP/CESM2/N_Amon_CESM2_amip-hist_combined_187001-201412.nc
82M	data/AMIP/CESM2/N_Amon_CESM2_amip-hist_combined_187001-201412.nc

CESM2 / amip-piForcing: 1 member(s)
    CDO: r1i1p1f1
  Saved: data/AMIP/CESM2/N_Amon_CESM2_amip-piForcing_combined_187001-201412.nc
512	data/AMIP/CESM2/N_Amon_CESM2_amip-piForcing_combined_187001-201412.nc

CIESM / amip-hist: 3 member(s)
    CDO: r1i1p1f1
    CDO: r2i1p1f1
    CDO: r3i1p1f1
  Saved: data/AMIP/CIESM/N_Amon_CIESM_amip-hist_combined_187001-201412.nc
82M	data/AMIP/CIESM/N_Amon_CIESM_amip-hist_combined_187001-201412.nc

CNRM-CM6-1 / amip-hist: 10 member(s)
    CDO: r1i1p1f2
    CDO: r2i1p1f2
    CDO: r3i1p1f2
    CDO: r4i1p1f2
    CDO: r5i1p1f2
    

In [17]:
import xarray as xr

bcc_file = Path(
    "./data/AMIP/BCC-CSM2-MR/"
    "N_Amon_BCC-CSM2-MR_amip-hist_combined_187001-201412.nc"
)

with xr.open_dataset(bcc_file) as ds:
    print(ds)
    print("Minimum N:", ds["N"].min().item())
    print("Maximum N:", ds["N"].max().item())

<xarray.Dataset> Size: 57MB
Dimensions:  (member: 1, time: 1740, lat: 64, lon: 128)
Coordinates:
  * member   (member) <U8 32B 'r1i1p1f1'
  * time     (time) object 14kB 1870-01-16 12:00:00 ... 2014-12-16 12:00:00
  * lat      (lat) float64 512B -87.86 -85.1 -82.31 -79.53 ... 82.31 85.1 87.86
  * lon      (lon) float64 1kB 0.0 2.812 5.625 8.438 ... 348.8 351.6 354.4 357.2
Data variables:
    N        (member, time, lat, lon) float32 57MB ...
Minimum N: -213.5305938720703
Maximum N: 198.5263671875


AttributeError: 'list' object has no attribute 'loc'